# 04 · Interpretación de los segmentos

En el reto 3 nos quedamos con 7 grupos de clientes. Ahora toca lo más importante: **entender quién hay en cada uno y qué haríamos con ellos**. Lo que pide el reto es un perfil de cada cluster que un equipo de marketing pueda usar, y un nombre descriptivo para cada uno.

Cómo lo vamos a hacer: siempre comparamos **cada segmento con el total de clientes**. Que un segmento tenga 48 meses de antigüedad no dice nada; que tenga 48 cuando la media es 34 ya es un rasgo.

Todas las cifras salen de `src/profiling.py`, que es lo mismo que usa el dashboard. Así es imposible que el notebook y el dashboard digan cosas distintas. Los nombres, frases y planes los escribimos nosotros en `src/segmentos.py`, pero sin números a mano: llevan huecos que se rellenan con el dato.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

from src import config, clean, clustering, profiling
from src.profiling import KPIS, fmt_kpi
from src.segmentos import SEGMENTOS, fill_texts, check_segmentos

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")
COLORES = config.SEGMENT_COLORS

## 1. Cargamos los clientes y su segmento

Usamos la tabla de clientes del reto 2 y las etiquetas que guardó el reto 3 (`outputs/labels_final.csv`). No re-entrenamos nada.

In [ ]:
clientes = clean.load_clients()
etiquetas = clustering.load_labels()

perfil = profiling.build_profile(clientes, etiquetas)
profiling.check_profile(perfil)  # los tamaños suman el total y cada distribución suma 100 %
check_segmentos(SEGMENTOS, perfil["meta"]["k"])  # cada segmento tiene nombre, frase y plan

base = perfil["base"]
segs = perfil["segmentos"]
nombres = [SEGMENTOS[s["id"]]["nombre"] for s in segs]
print(f"{perfil['meta']['n']:,} clientes en {perfil['meta']['k']} segmentos")

Antes de seguir comprobamos que el perfil cuadra con lo que vimos en el reto 3 (el "primer vistazo" de allí). Si alguien re-entrenara el modelo y cambiaran los grupos, esta celda fallaría y nos daríamos cuenta antes de presentar perfiles equivocados.

In [ ]:
assert [s["n"] for s in segs] == [41811, 31769, 31611, 27575, 15660, 15327, 9703]
antiguedad_03 = [47.99, 17.74, 32.25, 29.25, 17.36, 54.34, 39.09]
assert all(abs(s["kpis"]["antiguedad_meses"] - a) < 0.01 for s, a in zip(segs, antiguedad_03))
print("El perfil reproduce el reto 3")

## 2. Cuánto pesa cada segmento

Ya les ponemos los nombres que hemos decidido (en la sección 6 explicamos de dónde sale cada uno).

In [ ]:
tamanos = pd.DataFrame({"segmento": nombres, "clientes": [s["n"] for s in segs],
                        "%": [round(s["pct"], 1) for s in segs]})
tamanos

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(nombres[::-1], [s["pct"] for s in segs][::-1], color=COLORES[::-1], height=0.6)
for i, s in enumerate(segs[::-1]):
    ax.text(s["pct"] + 0.3, i, f"{s['pct']:.1f} %", va="center")
ax.set_xlabel("% de clientes")
ax.set_title("Tamaño de cada segmento")
ax.set_xlim(0, 28)
plt.tight_layout()
plt.savefig(config.FIG_DIR / "04_tamanos.png", dpi=110)
plt.show()

## 3. Qué distingue a cada segmento

Este es el gráfico más importante del notebook. Para cada segmento y cada indicador miramos **cuánto se separa de la media**, medido en desviaciones típicas (lo que en el reto 3 llamábamos z). Rojo es por encima de la media y azul por debajo; cuanto más intenso, más diferente. Dentro de cada casilla ponemos el valor real del segmento, para que se lea sin traducir.

In [ ]:
z = pd.DataFrame([s["z"] for s in segs], index=nombres)[list(KPIS)]
valores = pd.DataFrame([[fmt_kpi(k, s["kpis"][k]) for k in KPIS] for s in segs], index=nombres, columns=list(KPIS))
z.columns = valores.columns = [KPIS[k]["label"] for k in KPIS]

fig, ax = plt.subplots(figsize=(15, 5.5))
sns.heatmap(z, annot=valores, fmt="", cmap="RdBu_r", center=0, vmin=-2, vmax=2, linewidths=1,
            cbar_kws={"label": "diferencia con la media (desv. típicas)"}, annot_kws={"size": 8}, ax=ax)
ax.set_title("Qué distingue a cada segmento (rojo: por encima de la media, azul: por debajo)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.savefig(config.FIG_DIR / "04_mapa_calor.png", dpi=110)
plt.show()

Leyéndolo por filas, cada segmento tiene una o dos casillas muy marcadas que lo definen:

- **Fieles de taller:** las visitas al taller y la antigüedad, muy por encima.
- **Recién llegados:** antigüedad y edad del coche, muy por debajo.
- **Coche veterano:** la edad del coche, la más alta de todas.
- **Electrificados:** la electrificación, que es su seña de identidad.
- **Desconectados:** sin vehículos activos (todos).
- **Multivehículo:** nº de vehículos y vehículos activos.
- **Zonas premium:** el poder adquisitivo relativo. Es, junto con el "sin coche activo" de los Desconectados, la casilla más intensa del mapa.

Y hay columnas que casi no cambian de color, como la de coche conectado (solo destaca algo en Desconectados). Eso también es información: no es lo que separa a los clientes.

## 4. ¿Por qué no hay un segmento de furgonetas?

Es una pregunta que nos hicimos nosotros mismos, porque en el EDA vimos un bloque importante de furgonetas (Vito, Sprinter...). Miramos qué clase de coche lleva cada segmento frente al total:

In [ ]:
clase = pd.DataFrame({n: s["dist"]["clase"] for n, s in zip(nombres, segs)}).T
clase.loc["TOTAL"] = base["dist"]["clase"]
clase.round(1)

Las furgonetas están algo concentradas en Recién llegados y Multivehículo (alrededor del 20 %, frente al 11 % del total), pero en ningún segmento son mayoría. La clase de coche cambia poco de un segmento a otro.

La razón es una decisión que tomamos en el reto 2: las variables de tipo sí/no (clase de coche, motor, financiación) las dejamos en 0/1 sin escalar, para que no acapararan el modelo. Por eso pesan menos que las numéricas, y **los segmentos quedan definidos por la relación con la marca y el ciclo de vida del coche** (antigüedad, edad del coche, actividad, electrificación...), no por el producto.

Nos parece una segmentación más útil para marketing: dice **cuándo y cómo** hablarle a cada cliente (renovar, recuperar, fidelizar). El producto que se le ofrece se ajusta dentro de cada segmento con estas tablas.

## 5. Por dónde podemos hablarles

In [ ]:
canales = pd.DataFrame({n: s["canales"] for n, s in zip(nombres, segs)}).T
canales.loc["TOTAL"] = base["canales"]
canales.round(1)

Los consentimientos son prácticamente iguales en todos los segmentos: el email lo tiene casi todo el mundo, y teléfono y correo postal rondan el 85 %. Por eso no los metimos en el modelo (dicen por dónde contactar, no quién es el cliente). Para marketing la conclusión es sencilla: **el email llega a todos**, y el canal se elige por el tipo de mensaje, no por el segmento.

## 6. Ficha de cada segmento

Para cada segmento enseñamos lo mismo, en el mismo orden:

1. La ficha: nombre, tamaño, una frase de quién es, sus rasgos y el plan de acción en las cuatro palancas que pide el reto (oferta de vehículo, mantenimiento, fidelización y comunicación), más qué mediríamos para saber si funciona.
2. La tabla de indicadores: valor del segmento, valor del total y la diferencia.
3. El gráfico de diferencias con la media, ordenado de más a menos distinto.

Debajo de cada una explicamos por qué le hemos puesto ese nombre. Las reglas que seguimos para los nombres: dos o tres palabras, que describan el comportamiento y no la demografía, que se entiendan sin saber qué es K-Means y que no juzguen al cliente.

In [ ]:
PALANCAS = {"objetivo": "Objetivo", "oferta": "Oferta de vehículo", "mantenimiento": "Mantenimiento",
            "fidelizacion": "Fidelización", "comunicacion": "Comunicación", "medir": "Qué medimos"}


def ficha(seg_id):
    s = segs[seg_id]
    t = fill_texts(SEGMENTOS[seg_id], s)
    plan = "\n".join(f"| {PALANCAS[k]} | {v} |" for k, v in t["plan"].items())
    display(Markdown(
        f"### {t['nombre']} · {t['accion']}\n"
        f"**{s['n']:,} clientes ({s['pct']:.1f} %)**\n\n> {t['frase']}\n\n"
        f"**Rasgos:** {' · '.join(t['etiquetas'])}\n\n| Palanca | Qué hacemos |\n|---|---|\n{plan}"
    ))
    tabla = pd.DataFrame({
        "segmento": [fmt_kpi(k, s["kpis"][k]) for k in KPIS],
        "total": [fmt_kpi(k, base["kpis"][k]) for k in KPIS],
        "diferencia (desv.)": [round(s["z"][k], 2) for k in KPIS],
    }, index=[KPIS[k]["label"] for k in KPIS])
    display(tabla)

    zs = pd.Series(s["z"]).rename(lambda k: KPIS[k]["label"])
    zs = zs.reindex(zs.abs().sort_values().index)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(zs.index, zs.values, color=[COLORES[seg_id] if abs(v) >= 0.5 else config.BASE_GRAY for v in zs.values])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{t['nombre']}: diferencia con la media (desv. típicas)")
    plt.tight_layout()
    plt.savefig(config.FIG_DIR / f"04_ficha_{t['slug']}.png", dpi=110)
    plt.show()

In [ ]:
ficha(0)

**Por qué este nombre.** Lo que más les separa de la media son las **visitas al taller** (el doble que la media) y la **antigüedad** (unos 14 meses más). Son clientes que confían en el servicio oficial año tras año; de ahí "fieles de taller". Casi no tienen coches electrificados, que es la oportunidad que se les abre.

In [ ]:
ficha(1)

**Por qué este nombre.** Son los que **menos tiempo llevan** con la marca y los que tienen el **coche más nuevo**, con pocas visitas al taller todavía. "Recién llegados" lo dice tal cual. El reto con ellos es que la primera compra no sea la única.

In [ ]:
ficha(2)

**Por qué este nombre.** Su rasgo más fuerte es la **edad del coche**, la más alta de todos los segmentos (casi el doble que la media), y casi ninguno ha usado leasing. Es el grupo que tiene el cambio de coche más cerca, por eso "coche veterano" y la acción es renovar. Un aviso: en este segmento hay muchos clientes con la antigüedad por defecto de 36 meses que vimos en el EDA, así que probablemente llevan con nosotros más tiempo del que dice el dato.

In [ ]:
ficha(3)

**Por qué este nombre.** Aquí no hay duda: todos tienen un **eléctrico o un enchufable**, mientras que en el total es menos de uno de cada cuatro. Tienen coche reciente y son algo mayores que la media, lo que confirma lo que vimos en el EDA: quien se pasa al eléctrico no es el cliente más joven.

In [ ]:
ficha(4)

**Por qué este nombre.** Lo que les define es que **no tienen ningún vehículo activo**. Les llamamos "desconectados" y no "perdidos" a propósito: no tener coche activo no prueba que se hayan ido (pueden haberlo vendido, desvinculado de la app...). Pero es la señal de alarma más clara que tenemos, y además aquí se concentran los leasing terminados.

In [ ]:
ficha(5)

**Por qué este nombre.** Tienen **muchos más vehículos** que nadie y son los **clientes más antiguos**. Aquí caen casi todas las cuentas grandes (904 de las 908 con 20 o más vehículos), pero también familias con varios coches, por eso es el segmento más variado (el de silhouette más bajo en el reto 3). "Multivehículo" describe lo que tienen en común sin dar por hecho que todos son empresas.

In [ ]:
ficha(6)

**Por qué este nombre.** Su rasgo es, con diferencia, el **poder adquisitivo de su zona**: casi el doble del típico de su país. Son casi todos suizos, van mucho al taller y tienen más gama alta y SUV grandes que la media. Este segmento existe gracias a la variable relativa que creamos en el reto 2 (`ppi_rel`): sin ella, el poder adquisitivo solo habría dicho "es suizo".

## 7. Resumen

Los siete segmentos en una tabla, que es lo que nos llevaríamos a una reunión con marketing:

In [ ]:
resumen = pd.DataFrame({
    "segmento": nombres,
    "%": [round(s["pct"], 1) for s in segs],
    "acción": [SEGMENTOS[s["id"]]["accion"] for s in segs],
    "lo que le define": [", ".join(KPIS[k]["label"].lower() for k in s["rasgos"][:2]) for s in segs],
})
resumen

Tres avisos que hay que tener en la cabeza al usar estos segmentos:

- **"Sin coche activo" es una hipótesis de fuga, no un hecho.** Antes de lanzar una campaña de recuperación a los Desconectados conviene confirmar quién se ha ido de verdad.
- **La antigüedad de 36 meses es un valor por defecto** para un 12 % de los clientes (lo vimos en el EDA). Afecta sobre todo a Coche veterano.
- **Los segmentos hablan de la relación con la marca y del ciclo de vida del coche, no del producto.** La oferta concreta (furgoneta, SUV, gama alta) se ajusta dentro de cada segmento.

Todo esto, en versión interactiva, está en el dashboard (`outputs/dashboard_segmentos.html`).